<center><img src="./img/pong-thumbnail.png" width="200" alt="Skills Network Logo"  /></center>
  


## Proyect: Este proyecto trata sobre la creación de un sitio web para el poderoso concurso de Pong

##### Tiempo estimado necesario:  2 meses

El software ofrecerá una interfaz de usuario agradable y capacidades multijugador en tiempo real!

####  Requisitos Técnicos

- 
- 
- 
- 



## Paso 1: Configurar el entorno

Primero creamos una base sólida en contenedores para desplegar todos los servicios de nuestro proyecto. El orquestador que vamos a usar es `docker-compose.yml`

#### Emular Kubernetes (orquestador)

La idea original era usar el orquestador kunbernets por su arquitectura, flexibilidad y escalabiliad. Pero como no lo tenemos instalado en nuestra terminal, vamos a emular una arquitectura kubernets usando `docker-compose.yml`.

### ¿Cómo emulamos Kubernetes con docker-compose?

- Emulamos varios servicios y escalado (por ejemplo, varios contenedores de frontend y backend) usando docker-compose para definir múltiples instancias de los contenedores.
- Usaremos nginx o Traefik para gestionar el balanceo de carga.
Esto nos permitirá simular un entorno cercano a Kubernetes, aunque con limitaciones en temas como escalado automático.




In [ ]:
version: '3.8'

networks:
  transcendence:
    name: transcendence
    drivers: bridge

services:
  sqlite:   # Base de datos (SQLite)
    build: dockers/sqlite/.
    container_name: sqlite
    image: nouchka/sqlite3  # Imagen de SQLite para acceder a la base de datos (opcional)
    volumes:
      - sqlite_data:/var/lib/sqlite
    command: tail -f /dev/null  # Mantener el contenedor en ejecución
    networks:
      - transcendence

  backend:    # Backend (Node.js API)
    build: dockers/app/.
    container_name: app
    image: node:18-alpine
    working_dir: /usr/src/app
    volumes:
      - backend_data:/usr/src/app
    command: npm start
    ports:
      - "3000:3000"
    depends_on:
      - sqlite
    networks:
      - transcendence

  php:    # Servidor PHP (para componentes que aún lo requieran)
    build: dockers/php/.
    container_name: php
    image: php:8.1-fpm
    volumes:
      - php_data:/var/www/html
    networks:
      - transcendence

  frontend:   # Frontend (TypeScript + Tailwind)
    build: dockers/frontend/.
    container_name: frontend
    image: node:18-alpine
    volumes:
      - frontend_data:/usr/src/app
    command: npm run dev
    ports:
      - "8080:8080" # Puerto donde se servirá el frontend
    networks:
      - transcendence

  avalanche:      # Blockchain (Avalanche)
    build: dockers/blockchain/.
    container_name: blockchain
    image: avaplatform/avalanchego
    volumes:
      - blockchain_data:/root/.avalanchego
    ports:
      - "9650:9650"   # JSON RPC endpoint
      - "9651:9651"   # P2P port
    networks:
      - transcendence

  zap:    # Cyberseguridad (OWASP ZAP o similar)
    build: dockers/security/.
    container_name: security
    image: owasp/zap2docker-stable
    ports:
      - "8081:8081"
    networks:
      - transcendence   # Puerto para interfaz web de ZAP

volumes:
  sqlite_data:
    name: sqlite_data
    driver: local
    driver_opts:
      type: none
      device: "users/usuario/data/sqlite"
      o: bind
  
  backend_data:
    name: backend_data
    driver: local
    driver_opts:
      type: none
      device: "users/usuario/data/app"
      o: bind

  php_data:
    name: php_data
    driver: local
    driver_opts:
      type: none
      device: "users/usuario/data/php"
      o: bind
  
  frontend_data:
    name: frontend_data
    driver: local
    driver_opts:
      type: none
      device: "users/usuario/data/frontend"
      o: bind

  blockchain_data:
    name: blockchain_data
    driver: local
    driver_opts:
      type: none
      device: "users/usuario/data/blockchain"
      o: bind

  zap_data:
    name: zap_data
    driver: local
    driver_opts:
      type: none
      device: "users/usuario/data/security"
      o: bind


**Explicación:**
- `Base de datos (SQLite)`: Aunque SQLite no necesita un contenedor en sí, podemos usarlo para gestionar y persistir los datos, aunque también podría ser directamente gestionado por el backend.

- `Backend (Node.js)`: Un contenedor para Node.js que servirá a la API. Se conecta a SQLite usando el volumen compartido y expone el puerto 3000 para la API.

- `Servidor PHP`: Si necesitamos módulos PHP, aquí tenemos un contenedor para ejecutarlos.

- `Frontend (TypeScript + Tailwind)`: Un contenedor que servirá al frontend, donde podemos usar `npm run dev` para servirlo en desarrollo o cambiar a `npm run build` para producción.

- `Blockchain (Avalanche)`: El contenedor simula un nodo Avalanche, donde podemos interactuar con la blockchain y desplegar contratos en Solidity.

- `Cyberseguridad (OWASP ZAP)`: Se incluye OWASP ZAP para monitorear la seguridad de la aplicación en desarrollo.


# Paso 2: Configurar los contenedores

Creamos los contenedores y los almacenamos en los directorios: dockers.

## CONFIGURAR BACKEND:

**Estructura:**
```bash
dockers/backend/
│
├── data/               # Aquí se almacenará el archivo .db de SQLite
├── tools/
│   └── init.sql        # Script para inicializar la base de datos
├── app.js              # Aplicación Node.js
├── Dockerfile          # Dockerfile para la app Node.js
└── package.json        # Configuración del proyecto Node.js
```

### Guia paso a paso instalar la app Node.js.###

- 1. Instalar Node.js
Lo primero que necesitamos es instalar Node.js en la máquina. Verificar si ya lo tenemos instalado ejecutando el siguiente comando en tu terminal:
```bash
node -v
```
- 2. Si tienes Node.js instalado, deberías ver un número de versión. Si no, sigue estos pasos: Linux (Ubuntu/Debian):
```bash
sudo apt update
sudo apt install nodejs
sudo apt install npm
```
- 3. Crear proyecto Node.js<br/>
Crea un directorio:
```bash
mkdir backend
cd backend
```
Inicializa un proyecto de Node.js - Esto creará `package.json`, que es donde se definen las dependencias y la configuración del proyecto.:
```bash
npm init -y
```
Instalar las dependencias necesarias:

Necesitamos algunas dependencias básicas para la app, como el framework Express para crear la API. Ejecuta los siguientes comandos para instalar:

· Express: Un framework para manejar rutas y peticiones HTTP.<br>
· sqlite3: Para conectar y trabajar con SQLite.
```bash
npm install express sqlite3
```
Crear el archivo principal de la app:

Creamos un archivo que será el punto de entrada de la aplicación. Lo llamaremos app.js.
```bash
touch app.js
```
Escribir la lógica básica del servidor:

Abre app.js con tu editor de texto y escribe el siguiente código para configurar el servidor Node.js con Express y conectarlo a SQLite:







In [ ]:
const express = require('express');
const sqlite3 = require('sqlite3').verbose();
const fs = require('fs');
const path = require('path');

const app = express();
const port = 3000;

// Conectar a la base de datos SQLite
const dbPath = path.join(__dirname, 'data', 'sqlite.db');
const db = new sqlite3.Database(dbPath);

// Ejecutar el script de inicialización de la base de datos
const initSQL = fs.readFileSync(path.join(__dirname, 'tools', 'init.sql'), 'utf-8');
db.exec(initSQL, (err) => {
    if (err) {
        console.error('Error al inicializar la base de datos:', err.message);
    } else {
        console.log('Base de datos inicializada correctamente');
    }
});

// Ruta básica para probar el servidor
app.get('/', (req, res) => {
    res.send('¡Hola, mundo desde Node.js!');
});

// Iniciar el servidor
app.listen(port, () => {
    console.log(`Servidor escuchando en http://localhost:${port}`);
});


En este archivo, la aplicación de Node.js:

Se conecta a una base de datos SQLite.<br>
Ejecuta el script de inicialización `init.sql` desde la carpeta tools.<br>
Escucha en el puerto `3000` y responde a la ruta /.

Crear la estructura de directorios:

Crea la estructura de carpetas que necesitas para el proyecto, como tools y data:
```bash
mkdir tools
mkdir data
```
Luego, coloca tu archivo init.sql en el directorio tools.

Probar la app localmente:

Ahora, podemos ejecutar la aplicación para asegurarnos de que funciona correctamente.

In [ ]:
node app.js

Si todo está bien, deberías ver el mensaje en la consola:

In [ ]:
Servidor escuchando en http://localhost:3000

<br>

## CONFIGURAR SQLite:

1. Estructura para SQLite

SQLite almacena su base de datos en un archivo, así que no necesitamos un servidor separado. La base de datos simplemente se almacena en un archivo .db.

2. Archivo SQL para inicializar la base de datos:

En la carpeta tools, creamos el archivo init.sql, que contiene las instrucciones para crear las tablas necesarias y cualquier dato de inicio que se necesite.

In [ ]:
-- init.sql en tools/
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    password TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS games (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    player1_id INTEGER,
    player2_id INTEGER,
    winner_id INTEGER,
    FOREIGN KEY (player1_id) REFERENCES users (id),
    FOREIGN KEY (player2_id) REFERENCES users (id),
    FOREIGN KEY (winner_id) REFERENCES users (id)
);


3. Incluir SQLite en la app Node.js:

Ya hamos instalado sqlite3 en la aplicación Node.js, así que ahora solo necesitamos asegurarnos de que todo esté conectado.

Revisar archivo app.js para asegurar de que está conectando correctamente con SQLite y ejecutando el script init.sql cuando la app arranca.


4. Dockerfile para la app con SQLite

No necesitamos un servicio separado para SQLite, ya que se gestiona como parte de la aplicación Node.js.

El Dockerfile para Node.js así debería estar bien

In [ ]:
# Usar una imagen base de Node.js
FROM node:18-alpine

# Crear y establecer el directorio de trabajo
WORKDIR /usr/src/app

# Copiar el package.json y package-lock.json
COPY package*.json ./

# Instalar dependencias
RUN npm install

# Copiar el resto de los archivos de la app
COPY . .

# Crear la carpeta para la base de datos SQLite dentro del contenedor
RUN mkdir -p /usr/src/app/data

# Exponer el puerto de la app
EXPOSE 3000

# Comando para ejecutar la app
CMD ["node", "app.js"]


Este `Dockerfile` también asegura que se cree una carpeta data dentro del contenedor para almacenar la base de datos SQLite.

5. Inicializar la base de datos en el contenedor

La aplicación Node.js ejecuta automáticamente el script init.sql la primera vez que se conecta a la base de datos, por lo que cuando levantes los contenedores, la base de datos se inicializará correctamente dentro del contenedor.

## Authors


- David Gallego "davgalle"
- Nicolas Gonzalez de Mendoza "nicgonza"


Copyright © ErMichoss Corporation. All rights reserved.
